In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [1]:
import os
import shutil

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor



In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "vllm-sr/d3-edge"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 31 files: 100%|██████████████████████| 31/31 [00:00<00:00, 372.05it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/305 [00:00<?, ?it/s]

In [8]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
      

In [9]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1200,                  # High accuracy (Production grade)
    "nsamples": 512,                # More calibration data
    "batch_size": 4,
    "seqlen": 4096,
    "low_gpu_mem_usage": False,     # Keep on GPU for speed
    "enable_torch_compile": True,   # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [10]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-10-11 12:55:21 WARNING autoround.py L489: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-11 12:55:21 WARNING autoround.py L489: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-11 12:55:21 WARNING autoround.py L489: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [11]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-10-11 12:55:22 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.language_model.layers.[0-1,3-4,6-7].linear_attn.in_proj_a, model.language_model.layers.[0-1,3-4,6-7].linear_attn.in_proj_b
2026-10-11 12:55:22 INFO base.py L1365: `torch.compile` is enabled
2026-10-11 12:55:23 INFO orchestrator.py L594: start to cache block inputs
2026-10-11 12:55:23 INFO mllm.py L86: Using MLLM template: qwen3_5
2026-10-11 12:55:23 INFO mllm.py L125: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-10-11 12:55:23 INFO calib_dataset.py L1229: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-10-11 12:55:54 INFO device.py L1560: 'peak_ram': 7.12GB, 'peak_vram': 1.17GB
2026-10-11 12:55:54 INFO orchestrator.py L626: ca

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

packing: 100%|██████████| 90/90 [00:00<00:00, 128.97it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-11 13:01:37 INFO device.py L1560: 'peak_ram': 19.59GB, 'peak_vram': 18.31GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 768)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-11): 12 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=768, out_features=2304, bias=True)
             (proj): Linear(in_features=768, out_features=768, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
             (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
             (act_fn): GELUTanh(

In [12]:
# %%
# Files that must accompany the quantized backbone for downstream inference
auxiliary_files = [
    # Custom scripts & modeling code
    "modeling_d3.py",
    "pipeline_d3.py",
    "d3_engine.py",
    "d3_fast.py",
    "d3_format.py",
    "d3_kernels.py",
    "d3_runtime.py",
    "d3_server.py",
    # Dedicated readout head weights & configs
    "readout.safetensors",
    "decision_config.json",
    "MODEL_MANIFEST.json",
    # Templates & preprocessing configs
    "chat_template.jinja",
    "video_preprocessor_config.json",
    "requirements.txt",
]

# Note: Uses w4g32 to match your TUNING_CONFIG group_size: 32
export_subdirs = [
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"),
]

for export_dir in export_subdirs:
    if os.path.exists(export_dir):
        print(f"\n--> Injecting d3-edge custom runtime assets into: {export_dir}")
        for filename in auxiliary_files:
            src = os.path.join(LOCAL_PATH, filename)
            dst = os.path.join(export_dir, filename)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f"    Copied: {filename}")
            else:
                print(f"    Warning: Missing {src}")
    else:
        print(f"\n--> Target export directory not found: {export_dir}")


--> Injecting d3-edge custom runtime assets into: ./AutoRound/local_model-w4g32/auto-round-auto-gptq
    Copied: modeling_d3.py
    Copied: pipeline_d3.py
    Copied: d3_engine.py
    Copied: d3_fast.py
    Copied: d3_format.py
    Copied: d3_kernels.py
    Copied: d3_runtime.py
    Copied: d3_server.py
    Copied: readout.safetensors
    Copied: decision_config.json
    Copied: MODEL_MANIFEST.json
    Copied: chat_template.jinja
    Copied: video_preprocessor_config.json
    Copied: requirements.txt

--> Injecting d3-edge custom runtime assets into: ./AutoRound/local_model-w4g32/auto-gptq
    Copied: modeling_d3.py
    Copied: pipeline_d3.py
    Copied: d3_engine.py
    Copied: d3_fast.py
    Copied: d3_format.py
    Copied: d3_kernels.py
    Copied: d3_runtime.py
    Copied: d3_server.py
    Copied: readout.safetensors
    Copied: decision_config.json
    Copied: MODEL_MANIFEST.json
    Copied: chat_template.jinja
    Copied: video_preprocessor_config.json
    Copied: requirements.t

In [13]:
def push_to_hub(local_dir, repo_name, token):
    if not os.path.exists(local_dir):
        print(f"[Hub] Directory not found, skipping: {local_dir}")
        return

    full_repo_id = f"{HF_USER}/{repo_name}"
    try:
        api = HfApi()
        print(f"[Hub] Preparing repo: {full_repo_id}...")
        create_repo(full_repo_id, repo_type="model", private=False, token=token, exist_ok=True)
        
        print(f"[Hub] Uploading folder from {local_dir}...")
        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            token=token
        )
        print(f"[Hub] Upload complete: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] Error uploading {full_repo_id}: {e}")


In [14]:

base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()


In [ ]:

if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token
    )

else:
    print("No Hugging Face token found. Skipping upload.")

[Hub] Preparing repo: Vishva007/d3-edge-W4A16-AutoRound...


[Hub] Uploading folder from ./AutoRound/local_model-w4g32/auto-round-auto-gptq...
[Hub] Upload complete: https://huggingface.co/Vishva007/d3-edge-W4A16-AutoRound
[Hub] Preparing repo: Vishva007/d3-edge-W4A16-AutoRound-GPTQ...
[Hub] Uploading folder from ./AutoRound/local_model-w4g32/auto-gptq...
[Hub] Upload complete: https://huggingface.co/Vishva007/d3-edge-W4A16-AutoRound-GPTQ
[Hub] Preparing repo: Vishva007/d3-edge-W4A16-AutoRound-LLM-Compressor...
[Hub] Uploading folder from ./AutoRound/local_model-w4g32/llm-compressor-wint-a16...
[Hub] Upload complete: https://huggingface.co/Vishva007/d3-edge-W4A16-AutoRound-LLM-Compressor
